# 2.2 · El bucle desde cero

**Objetivo:** programar el bucle agente completo y ver al modelo **encadenar** decisiones.

**Al terminar tendrás:** una función `ejecutar_agente()` con dos herramientas, trazas y límite de iteraciones.

Un agente es un bucle `while`. Literalmente. Vamos a escribirlo.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

from herramientas import buscar_incidencias, crear_incidencia, restaurar_datos

In [ ]:
SYSTEM = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, de forma breve y basándote solo en los datos que tengas."
)

## Segunda herramienta

Añadimos `crear_incidencia`. Todos sus campos son obligatorios y la prioridad solo admite tres valores (`enum`).

La descripción no solo dice qué hace: también dice **cuándo no usarla**.

In [ ]:
TOOLS = [
    {
        "name": "buscar_incidencias",
        "description": (
            "Busca incidencias de soporte en la base de datos. Úsala siempre que el usuario pregunte "
            "por incidencias existentes, su estado o las de un cliente concreto. Devuelve una lista "
            "de incidencias con id, cliente, asunto, estado y prioridad. Si no hay coincidencias devuelve []."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "texto":   {"type": "string", "description": "Texto libre a buscar en asunto y descripción"},
                "estado":  {"type": "string", "enum": ["abierta", "cerrada"]},
                "cliente": {"type": "string", "description": "Nombre exacto del cliente"},
            },
            "required": [],
        },
    },
    {
        "name": "crear_incidencia",
        "description": (
            "Crea una incidencia de soporte nueva y devuelve la incidencia creada con su id. "
            "Úsala solo cuando no exista ya una incidencia abierta sobre el mismo problema "
            "para el mismo cliente: compruébalo antes con buscar_incidencias."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cliente":     {"type": "string", "description": "Nombre exacto del cliente"},
                "asunto":      {"type": "string", "description": "Resumen del problema en menos de 10 palabras"},
                "descripcion": {"type": "string", "description": "Qué ocurre, desde cuándo y a quién afecta"},
                "prioridad":   {"type": "string", "enum": ["alta", "media", "baja"]},
            },
            "required": ["cliente", "asunto", "descripcion", "prioridad"],
        },
    },
]

El modelo pide herramientas por nombre. Un diccionario traduce cada nombre a la función de Python que hay que ejecutar:

In [ ]:
FUNCIONES = {"buscar_incidencias": buscar_incidencias, "crear_incidencia": crear_incidencia}

## El bucle

Es la vuelta del 2.1 metida en un `for`: llamar → si pide herramientas, ejecutarlas y devolver los resultados → volver a llamar. Termina cuando el modelo responde sin pedir nada.

In [ ]:
MAX_ITERACIONES = 10

def ejecutar_agente(pregunta, messages):
    messages.append({"role": "user", "content": pregunta})
    for i in range(MAX_ITERACIONES):
        response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                          tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")

        resultados = []
        for bloque in response.content:
            if bloque.type != "tool_use":
                continue
            print(f"  [iter {i}] → {bloque.name}({bloque.input})")
            try:
                salida = FUNCIONES[bloque.name](**bloque.input)
                resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                                   "content": json.dumps(salida, ensure_ascii=False)})
            except Exception as e:
                resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                                   "content": f"Error: {e}", "is_error": True})
        messages.append({"role": "user", "content": resultados})

    raise RuntimeError("Se alcanzó el máximo de iteraciones")

Cuatro decisiones de diseño, una por línea importante:

1. Se añade **todo** `response.content` al historial (incluido `thinking`), no solo el texto.
2. Una respuesta puede traer **varios `tool_use`**: todos los `tool_result` van juntos en **un solo** mensaje `user`.
3. Los errores **se devuelven al modelo** con `is_error` en vez de reventar: así puede corregirse.
4. `MAX_ITERACIONES`: un agente sin límite es un bucle infinito con factura.

`messages` se pasa **desde fuera**: al terminar podemos inspeccionar en otra celda todo lo que ha ocurrido.

In [ ]:
historial = []
ejecutar_agente("¿Cuántas incidencias abiertas tiene ACME?", historial)

### La radiografía del agente

El tipo de cada bloque de cada mensaje. Los mensajes `user` que escribimos nosotros llevan un string; los demás, listas de bloques (objetos del SDK o diccionarios).

In [ ]:
def tipos(content):
    if isinstance(content, str):
        return ["text"]
    return [b["type"] if isinstance(b, dict) else b.type for b in content]

for m in historial:
    print(m["role"], tipos(m["content"]))

## El modelo decide la ruta

Una petición que necesita **encadenar** herramientas. Antes, dejamos `incidencias.json` en su estado inicial (la demo crea incidencias).

In [ ]:
restaurar_datos()

In [ ]:
PREGUNTA = (
    "El cliente Globex dice que no puede exportar a PDF. Si ya hay una incidencia abierta "
    "sobre eso, dime cuál; si no, créala con prioridad media."
)

In [ ]:
historial = []
print(ejecutar_agente(PREGUNTA, historial))

En las trazas: `buscar_incidencias` → no encuentra nada → `crear_incidencia`.

Repetimos **la misma pregunta**. Ahora la incidencia existe:

In [ ]:
historial = []
print(ejecutar_agente(PREGUNTA, historial))

Esta vez la encuentra y **no** crea un duplicado. Nuestro código es el mismo: la ruta la decide el modelo.

## Romperlo para entenderlo

Cada fallo es una copia modificada de la función, para no estropear la buena.

### Fallo 1: sin el turno `assistant`

Quitamos la línea que guarda la respuesta del modelo en el historial.

In [ ]:
# ❌ FALLO PROVOCADO
def ejecutar_agente_sin_assistant(pregunta, messages):
    messages.append({"role": "user", "content": pregunta})
    for i in range(MAX_ITERACIONES):
        response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                          tools=TOOLS, messages=messages)
        # messages.append({"role": "assistant", "content": response.content})   ← quitada

        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")

        resultados = []
        for bloque in response.content:
            if bloque.type != "tool_use":
                continue
            print(f"  [iter {i}] → {bloque.name}({bloque.input})")
            salida = FUNCIONES[bloque.name](**bloque.input)
            resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                               "content": json.dumps(salida, ensure_ascii=False)})
        messages.append({"role": "user", "content": resultados})

ejecutar_agente_sin_assistant("¿Cuántas incidencias abiertas tiene ACME?", [])

Error 400: el `tool_result` responde a un `tool_use` que no está en el historial. Para la API, ese resultado no responde a nada.

### Fallo 2: un `tool_use_id` equivocado

In [ ]:
# ❌ FALLO PROVOCADO
def ejecutar_agente_id_erroneo(pregunta, messages):
    messages.append({"role": "user", "content": pregunta})
    for i in range(MAX_ITERACIONES):
        response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                          tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")

        resultados = []
        for bloque in response.content:
            if bloque.type != "tool_use":
                continue
            print(f"  [iter {i}] → {bloque.name}({bloque.input})")
            salida = FUNCIONES[bloque.name](**bloque.input)
            resultados.append({"type": "tool_result", "tool_use_id": "toolu_inventado",   # ←
                               "content": json.dumps(salida, ensure_ascii=False)})
        messages.append({"role": "user", "content": resultados})

ejecutar_agente_id_erroneo("¿Cuántas incidencias abiertas tiene ACME?", [])

Error 400 otra vez: cada `tool_use` necesita su `tool_result` con **el mismo id**, justo en el mensaje siguiente.

### Fallo 3: una herramienta que falla

Simulamos que la base de datos está bloqueada la primera vez que se intenta crear una incidencia. Esta vez usamos la función buena: el error se devuelve al modelo con `is_error`.

In [ ]:
# ❌ FALLO PROVOCADO
intentos = 0

def crear_incidencia_inestable(**kwargs):
    global intentos
    intentos += 1
    if intentos == 1:
        raise RuntimeError("la base de datos está bloqueada, vuelve a intentarlo")
    return crear_incidencia(**kwargs)

FUNCIONES["crear_incidencia"] = crear_incidencia_inestable
restaurar_datos()
print(ejecutar_agente(PREGUNTA, []))
FUNCIONES["crear_incidencia"] = crear_incidencia   # dejamos la buena

El modelo lee el error y vuelve a intentarlo. Si el bucle hubiera reventado con la excepción, no habría tenido esa oportunidad.

## Resumen

- Un agente es un bucle: llamar, ejecutar lo que pide, devolver resultados, repetir hasta que no pida nada.
- Las cuatro decisiones: guardar todo `response.content`, todos los `tool_result` en un solo mensaje, errores devueltos con `is_error` y un límite de iteraciones.
- La ruta (buscar y luego crear, o solo buscar) la decide el modelo, no el código.

Fíjate en cuántas cosas frágiles hay: ids, orden de mensajes, errores, límites. Recuérdalas: en el bloque 3 vamos a ver cómo desaparecen.

## 🔸 Reto

Añade una tercera herramienta `cerrar_incidencia(id)` y pide al agente que cierre todas las incidencias de baja prioridad de ACME.

La función de negocio ya existe: `from herramientas import cerrar_incidencia`.

### Solución

Tres cambios: la definición en `TOOLS`, la entrada en `FUNCIONES` y nada más. El bucle no cambia: es genérico.

In [ ]:
from herramientas import cerrar_incidencia

TOOLS.append({
    "name": "cerrar_incidencia",
    "description": (
        "Cierra una incidencia abierta a partir de su id (formato INC-123). "
        "Si no conoces el id, búscalo antes con buscar_incidencias."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"id": {"type": "string", "description": "Id de la incidencia, p. ej. INC-002"}},
        "required": ["id"],
    },
})
FUNCIONES["cerrar_incidencia"] = cerrar_incidencia

In [ ]:
restaurar_datos()
historial = []
print(ejecutar_agente("Cierra todas las incidencias de baja prioridad de ACME que estén abiertas.", historial))

Comprobamos el resultado directamente en los datos:

In [ ]:
buscar_incidencias(cliente="ACME")

Lo dejamos todo como estaba:

In [ ]:
restaurar_datos()